In [3]:
import snapatac2 as snap
import numpy as np
import polars as pl
import scanpy as sc
snap.__version__

'2.5.3'

In [4]:
# 定义函数，筛选出显著富集的 top_K TFs
import polars as pl
def select_significant_top_TFs(motifs_results, K, padj_cutoff):
    result = {}
    for cluster, df in motifs_results.items():
        # 1. 筛选显著 TF
        df_sig = df.filter(pl.col("adjusted p-value") <= padj_cutoff)

        # 2. 如果一个簇显著 TF 不足 K 个，就自动保留所有现有的
        df_top = (df_sig.sort("log2(fold change)", descending=True).head(K))

        # 保存结果
        result[cluster] = df_top
    return result

In [5]:
import networkx as nx

def build_CER_TF_outflow_network(outflows_top_TFs, CER_top_genes, selected_CERs):
    """
    构建 TF -> CER 的有向图，边权重为 TF 对 CER 的绝对贡献。

    Parameters
    ----------
    outflows_top_TFs : dict
        每个键是 outflow 名称，对应值是包含 TF 的 DataFrame（Polars 或 Pandas），列名为 "name"。
    CER_top_genes : dict
        每个键是 CER 名称，对应值是包含基因贡献信息的 DataFrame，列名至少包括 "gene" 和 "abs_importance"。
    selected_CERs : list of str
        需要考虑的 CER 键名。

    Returns
    -------
    G : nx.DiGraph
        构建好的有向图，节点是 TF 和 CER，边的权重为 abs_importance。
    """
    G = nx.DiGraph()

    for outflow, tf_df in outflows_top_TFs.items():
        tf_df = tf_df.to_pandas()
        # Polars → Python list
        tfs = tf_df["name"].to_list()
        for tf in tfs:
            for cer in selected_CERs:
                if cer not in CER_top_genes:
                    continue
                cer_df = CER_top_genes[cer]

                # Polars → Pandas
                if hasattr(cer_df, "to_pandas"):
                    cer_pd = cer_df.to_pandas()
                else:
                    cer_pd = cer_df

                # 查找 TF 是否在 CER 中
                row = cer_pd[cer_pd["gene"] == tf]
                if row.empty:
                    continue
                importance = float(row["abs_importance"].iloc[0])
                tf_log_fold_change = float(tf_df[tf_df['name']==tf]['log2(fold change)'])
                # 添加边
                G.add_edge(cer, tf, weight=importance)
                G.add_edge(tf, outflow, weight=tf_log_fold_change)

    return G


In [6]:
# 加载对 CERs 贡献较高的 top 100 基因
import pickle
with open("contributions_to_target_CERs_top100.pkl", "rb") as f:
    CER_top_genes = pickle.load(f)

In [7]:
# 加载经过 scglue 分析得到的与 outflows 关联性较强的 Peaks 数据
with open("Edn1_peaks_to_outflows.pkl", "rb") as f:
    connect_peaks = pickle.load(f)
connect_peaks

{'Edn1': Index(['chr13:42300994-42301907'], dtype='object'),
 'Mdk': Index(['chr2:91931310-91932173'], dtype='object'),
 'Fgf3': Index(['chr7:144837650-144838395', 'chr7:144838796-144839572'], dtype='object'),
 'Sema3c': Index(['chr5:17574171-17575065', 'chr5:17586414-17587249',
        'chr5:17587460-17588293', 'chr5:17637105-17638171',
        'chr5:17698651-17699586'],
       dtype='object'),
 'Wnt7b': Index(['chr15:85539558-85540248', 'chr15:85564684-85565344',
        'chr15:85569912-85570711', 'chr15:85582933-85583862'],
       dtype='object'),
 'Wnt7a': Index(['chr6:91371980-91372878'], dtype='object'),
 'Sema3g': Index(['chr14:31215276-31216257', 'chr14:31217439-31218331'], dtype='object'),
 'Fgf9': Index(['chr14:58112108-58112856'], dtype='object'),
 'Rarres2': Index(['chr6:48572559-48573400'], dtype='object')}

In [8]:
# 使用snapatac2，在这些Peaks上进行motifs富集分析
outflows_motifs = snap.tl.motif_enrichment(
    motifs=snap.datasets.cis_bp(unique=True),
    regions=connect_peaks,
    genome_fasta=snap.genome.mm10
)

2025-12-01 04:11:27 - INFO - Fetching 18 sequences ...
2025-12-01 04:11:29 - INFO - Computing enrichment ...
100%|██████████| 1165/1165 [00:45<00:00, 25.49it/s]


In [9]:
# 接下来筛选出显著的 top TFs
outflows_top_TFs = select_significant_top_TFs(outflows_motifs, K=60, padj_cutoff=0.05)

In [10]:
# 将 TFs 的名字改为第一个字母大写，其余字母小写，和 RNA 数据中的基因名写法保持一致
import polars as pl
for key, df in outflows_top_TFs.items():
    outflows_top_TFs[key] = (
        df
        .with_columns(
            # 先全部小写
            pl.col("name").str.to_lowercase().alias("name")
        )
        .with_columns(
            # 大写首字母 + 保留其余
            (pl.col("name").str.slice(0, 1).str.to_uppercase()
             + pl.col("name").str.slice(1, None)).alias("name")
        )
    )
outflows_top_TFs

{'Edn1': shape: (60, 6)
 ┌────────────────────┬────────┬────────┬───────────────────┬─────────┬──────────────────┐
 │ id                 ┆ name   ┆ family ┆ log2(fold change) ┆ p-value ┆ adjusted p-value │
 │ ---                ┆ ---    ┆ ---    ┆ ---               ┆ ---     ┆ ---              │
 │ str                ┆ str    ┆ f32    ┆ f64               ┆ f64     ┆ f64              │
 ╞════════════════════╪════════╪════════╪═══════════════════╪═════════╪══════════════════╡
 │ BBX+M05764_2.00    ┆ Bbx    ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ BPTF+M11435_2.00   ┆ Bptf   ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ DMRT2+M10434_2.00  ┆ Dmrt2  ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ DPF1+M04389_2.00   ┆ Dpf1   ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ …                  ┆ …      ┆ …      ┆ …                 ┆ …       ┆ …                │
 │ RREB1+M10203_2.00  ┆ Rreb1  ┆ null   ┆ 2.169925          ┆ 0.0 

In [13]:
with open("network_inflow_Edn1.pkl", "rb") as f:
    network_inflow = pickle.load(f)

In [14]:
CERs = [cer for cer in network_inflow.nodes if cer.startswith("CER-")]


In [15]:
inflow_G = build_CER_TF_outflow_network(outflows_top_TFs, CER_top_genes, CERs)
# 查看边信息
print(inflow_G.edges(data=True))

/tmp/ipykernel_562124/3802510987.py:44: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  tf_log_fold_change = float(tf_df[tf_df['name']==tf]['log2(fold change)'])


[('CER-18', 'Sox11', {'weight': 0.005625126883387566}), ('CER-18', 'Nfia', {'weight': 0.0035910687875002623}), ('CER-18', 'Zeb2', {'weight': 0.00652696006000042}), ('Sox11', 'Edn1', {'weight': 4.169925001442312}), ('CER-5', 'Sox11', {'weight': 0.003672396531328559}), ('CER-5', 'Nfia', {'weight': 0.003090118756517768}), ('CER-5', 'Rarb', {'weight': 0.002565372036769986}), ('CER-5', 'Zeb2', {'weight': 0.0036976963747292757}), ('CER-1', 'Sox11', {'weight': 0.00442655710503459}), ('CER-1', 'Nfia', {'weight': 0.004610129632055759}), ('CER-1', 'Rbpj', {'weight': 0.0027475226670503616}), ('CER-1', 'Zeb2', {'weight': 0.006081235129386187}), ('CER-15', 'Sox11', {'weight': 0.022258887067437172}), ('CER-15', 'Nfia', {'weight': 0.019207745790481567}), ('CER-15', 'Zeb2', {'weight': 0.018177682533860207}), ('Nfia', 'Sema3c', {'weight': 1.8479969065549502}), ('Rarb', 'Wnt7b', {'weight': 2.169925001442312}), ('Rbpj', 'Sema3g', {'weight': 2.169925001442312}), ('Zeb2', 'Sema3g', {'weight': 2.16992500144

In [16]:
with open("inflow-Edn1_CER_TF_outflow_network.pkl", "wb") as f:
    pickle.dump(inflow_G, f)

In [17]:
with open("network_outflow_Edn1.pkl", "rb") as f:
    network_outflow = pickle.load(f)

In [18]:
CERs = [cer for cer in network_outflow.nodes if cer.startswith("CER-")]
CERs

['CER-1', 'CER-5']

In [20]:
top_TFs = {'Edn1': outflows_top_TFs['Edn1']}
top_TFs

{'Edn1': shape: (60, 6)
 ┌────────────────────┬────────┬────────┬───────────────────┬─────────┬──────────────────┐
 │ id                 ┆ name   ┆ family ┆ log2(fold change) ┆ p-value ┆ adjusted p-value │
 │ ---                ┆ ---    ┆ ---    ┆ ---               ┆ ---     ┆ ---              │
 │ str                ┆ str    ┆ f32    ┆ f64               ┆ f64     ┆ f64              │
 ╞════════════════════╪════════╪════════╪═══════════════════╪═════════╪══════════════════╡
 │ BBX+M05764_2.00    ┆ Bbx    ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ BPTF+M11435_2.00   ┆ Bptf   ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ DMRT2+M10434_2.00  ┆ Dmrt2  ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ DPF1+M04389_2.00   ┆ Dpf1   ┆ null   ┆ 4.169925          ┆ 0.0     ┆ 0.0              │
 │ …                  ┆ …      ┆ …      ┆ …                 ┆ …       ┆ …                │
 │ RREB1+M10203_2.00  ┆ Rreb1  ┆ null   ┆ 2.169925          ┆ 0.0 

In [21]:
outflow_G = build_CER_TF_outflow_network(top_TFs, CER_top_genes, CERs)
# 查看边信息
print(outflow_G.edges(data=True))

[('CER-1', 'Sox11', {'weight': 0.00442655710503459}), ('Sox11', 'Edn1', {'weight': 4.169925001442312}), ('CER-5', 'Sox11', {'weight': 0.003672396531328559})]


/tmp/ipykernel_562124/3802510987.py:44: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  tf_log_fold_change = float(tf_df[tf_df['name']==tf]['log2(fold change)'])


In [22]:
with open("Edn1_CER_TF_outflow_network.pkl", "wb") as f:
    pickle.dump(outflow_G, f)